In [ ]:
import numpy as np

from magtense.magstatics import Tiles, run_simulation
from magtense.micromag import MicromagProblem
from magtense.utils import create_plot

In [ ]:
# Geometry Setup (240nm bounding box)
LLC = np.array([0, 0, 0], dtype=np.float64)
URC = np.array([1, 1, 1], dtype=np.float64) * 240e-9 / 10

index = 1
ms_index = 1

URC[index] *= 2

box_dim = URC - LLC
dx, dy, dz = box_dim
num_x, num_y, num_z = 1, 1, 1

if index == 0:
    dx /= 2
    num_x = 2
elif index == 1:
    dy /= 2
    num_y = 2
elif index == 2:
    dz /= 2
    num_z = 2

print(f"Box dimensions: {box_dim}")
print(f"dx: {dx}, dy: {dy}, dz: {dz}")


In [ ]:
# (Grid Positions)
x_c = np.linspace(LLC[0] + dx/2, URC[0] - dx/2, num_x)
y_c = np.linspace(LLC[1] + dy/2, URC[1] - dy/2, num_y)
z_c = np.linspace(LLC[2] + dz/2, URC[2] - dz/2, num_z)

X, Y, Z = np.meshgrid(x_c, y_c, z_c, indexing='ij')
offsets = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
print(f"Offsets:\n{offsets}")

In [ ]:
# define problem - simular to grain, but set K0 and A0 to zero to isolate the effect of the demag field
mu0 = 4 * np.pi * 1e-7
Ms = 1.61/mu0 ; # 1.61 T
alpha = 4000.0
gamma = 0.0
K0 = 0.0 #4.3e6 ;    # J/m3
A0 = 0.0 #7.7e-12 ;  # J/m3
res = np.array([2, 1, 1], dtype=int)

if index == 0:
    res = np.array([2, 1, 1], dtype=int)
elif index == 1:
    res = np.array([1, 2, 1], dtype=int)
elif index == 2:
    res = np.array([1, 1, 2], dtype=int)

grid_L = box_dim
grid_pts = offsets
grid_abc = np.tile([dx, dy, dz], (2, 1))
#grid_type : str = "unstructuredPrisms"
problem = MicromagProblem(
    res=res,
    grid_L=grid_L,
    alpha=alpha,
    #grid_pts=grid_pts,
    #grid_abc=grid_abc,
    cuda=True,
    cvode=False,
   #grid_type=grid_type,
)
problem.Ms = Ms * np.ones((2, 1)) 
problem.Ms[ms_index] *= 10
problem.K0 = K0 * np.ones((2, 1))
problem.A0 = A0 * np.ones((2, 1))
problem.m0 = np.array([[1, 0, 0], [0, 0, 1]], dtype=np.float64)  
problem.u_ea = np.array([[0, 0, 0], [0, 0, 0]], dtype=np.float64)


In [ ]:
tiles = Tiles(
    n=2,
    M_rem=problem.Ms,
    tile_type=[2,2],
    color=[[1, 0, 0], [0, 0, 1]]
)
tiles.size = grid_abc
tiles.offset = grid_pts
tiles.M = problem.m0
create_plot(tiles)

In [ ]:
t_end = 1e-9
nt = 2

nt_h_ext = 2
def h_ext_fct(t) -> np.ndarray:
    return np.array([0, 0, 0])

out = problem.run_simulation(
            t_end=t_end, nt=nt, fct_h_ext=h_ext_fct, nt_h_ext=2
        )

In [ ]:
tiles = Tiles(
    n=2,
    M_rem=problem.Ms,
    tile_type=[2,2],
    color=[[1, 0, 0], [0, 0, 1]]
)
tiles.size = grid_abc
tiles.offset = grid_pts
tiles.M[0] = out[1][1][0][0]
tiles.M[1] = out[1][1][1][0]
create_plot(tiles)

In [ ]:
out[1][1]